#### REAL-TIME GESTURE DETECTOR

### 0. Data Collection

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import os
from datetime import datetime

GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "low_punch", "high_punch", "high_kick", "strong_kick",
    "hit_combo", "neutral"
]

FRAMES_PER_CLIP = 60   # 1 second at ~30fps
CLIPS_PER_GESTURE = 5

OUTPUT_DIR = "karate_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for g in GESTURES:
    os.makedirs(os.path.join(OUTPUT_DIR, g), exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

cap = cv2.VideoCapture(0)

def capture_clip(label, clip_no):
    keypoints_list = []
    print(f"\nPrepare for {label} clip {clip_no+1}")
    for i in range(3, 0, -1):
        print(f"Starting in {i}...")
        cv2.waitKey(1000)

    print("Recording...")
    frame_count = 0
    while frame_count < FRAMES_PER_CLIP:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose.process(rgb)

        if result.pose_landmarks:
            kps = [[lm.x, lm.y, lm.z] for lm in result.pose_landmarks.landmark]
            keypoints_list.append(kps)
        else:
            keypoints_list.append(np.zeros((33,3)))

        cv2.putText(frame, f"{label} clip {clip_no+1}", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
        cv2.imshow("Recording", frame)
        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    np.save(os.path.join(OUTPUT_DIR, label, f"{timestamp}.npy"), np.array(keypoints_list))
    print(f"Saved {label} clip {clip_no+1}")

# Collect all gestures
for g in GESTURES:
    for clip_no in range(CLIPS_PER_GESTURE):
        capture_clip(g, clip_no)

cap.release()
cv2.destroyAllWindows()
print("Data collection complete.")



Prepare for jump clip 1
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 1

Prepare for jump clip 2
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 2

Prepare for jump clip 3
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 3

Prepare for jump clip 4
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 4

Prepare for jump clip 5
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved jump clip 5

Prepare for crouch clip 1
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 1

Prepare for crouch clip 2
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 2

Prepare for crouch clip 3
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 3

Prepare for crouch clip 4
Starting in 3...
Starting in 2...
Starting in 1...
Recording...
Saved crouch clip 4

Prepare for crouch c

### 1. Prepare the dataset for training

In [12]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "karate_data"
GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "low_punch", "high_punch", "high_kick", "strong_kick",
    "hit_combo", "neutral"
]

X = []
y = []

for g in GESTURES:
    folder = os.path.join(DATA_DIR, g)
    for file in os.listdir(folder):
        arr = np.load(os.path.join(folder, file))  # shape: (30,33,3)
        X.append(arr.flatten())  # flatten to 1D vector
        y.append(g)

X = np.array(X)
y = np.array(y)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


### 2. Train a Decision tree classifier

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

clf = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

clf.fit(X_train, y_train)

labels = list(range(len(le.classes_)))
y_pred = clf.predict(X_test) 

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, labels=labels, target_names=le.classes_))

joblib.dump(clf, "karate_dt_model.pkl")
joblib.dump(le, "label_encoder.pkl")


Test Accuracy: 0.7
              precision    recall  f1-score   support

      crouch       0.00      0.00      0.00         0
   high_kick       1.00      0.50      0.67         2
  high_punch       1.00      1.00      1.00         2
   hit_combo       0.00      0.00      0.00         0
        jump       0.00      0.00      0.00         0
   low_punch       0.00      0.00      0.00         0
   move_left       1.00      1.00      1.00         1
  move_right       0.40      1.00      0.57         2
     neutral       1.00      0.50      0.67         2
 strong_kick       0.00      0.00      0.00         1

    accuracy                           0.70        10
   macro avg       0.44      0.40      0.39        10
weighted avg       0.78      0.70      0.68        10



c:\Users\wakde\OneDrive\Desktop\Body-Game-Gesture-Detection\AILab\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\wakde\OneDrive\Desktop\Body-Game-Gesture-Detection\AILab\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\wakde\OneDrive\Desktop\Body-Game-Gesture-Detection\AILab\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to contro

['label_encoder.pkl']

### 3. Live gesture detection

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import joblib

# Load model and label encoder
clf = joblib.load("karate_dt_model.pkl")
le = joblib.load("label_encoder.pkl")

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
cap = cv2.VideoCapture(0)

SEQUENCE_LENGTH = 60  # frames per clip
buffer = deque(maxlen=SEQUENCE_LENGTH)

print("Press ESC or Q to quit...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = pose.process(rgb)

    gesture = "NEUTRAL"
    if res.pose_landmarks:
        # Convert landmarks to list of [x,y,z]
        lm_list = [[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark]
        buffer.append(lm_list)

        if len(buffer) == SEQUENCE_LENGTH:
            clip_arr = np.array(buffer).flatten().reshape(1, -1)
            pred = clf.predict(clip_arr)
            gesture = le.inverse_transform(pred)[0]

        # Draw skeleton
        mp.solutions.drawing_utils.draw_landmarks(frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    cv2.putText(frame, gesture, (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
    cv2.imshow("Karate Fighter Gesture Detection", frame)

    key = cv2.waitKey(1)
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press ESC or Q to quit...
